# Agentic SDLC Stack Explorer
Pre-wired connections to every service in the stack.
Run cells top-to-bottom on first use to verify everything is reachable.

In [1]:
# ── Imports & config ─────────────────────────────────────────────────────────
import os, json, requests, textwrap
from datetime import datetime

LITELLM_URL  = os.environ.get('LITELLM_BASE_URL', 'http://litellm:4000')
OLLAMA_URL   = os.environ.get('OLLAMA_BASE_URL',  'http://ollama:11434')
N8N_URL      = os.environ.get('N8N_BASE_URL',     'http://n8n:5678')
DB_URL       = os.environ.get('DATABASE_URL', '')
LITELLM_KEY  = os.environ.get('LITELLM_API_KEY', '')

HEADERS = {'Authorization': f'Bearer {LITELLM_KEY}', 'Content-Type': 'application/json'}

print('Config loaded')
print(f'  LiteLLM : {LITELLM_URL}')
print(f'  Ollama  : {OLLAMA_URL}')
print(f'  n8n     : {N8N_URL}')

Config loaded
  LiteLLM : http://litellm:4000
  Ollama  : http://ollama:11434
  n8n     : http://n8n:5678


## 1. Service Health Checks

In [2]:
# Check all services are reachable
services = {
    'LiteLLM health':  f'{LITELLM_URL}/health/liveliness',
    'LiteLLM models':  f'{LITELLM_URL}/v1/models',
    'Ollama version':  f'{OLLAMA_URL}/api/version',
    'Ollama models':   f'{OLLAMA_URL}/api/tags',
    'n8n health':      f'{N8N_URL}/healthz',
}

for name, url in services.items():
    try:
        r = requests.get(url, headers=HEADERS, timeout=5)
        status = '✅' if r.status_code == 200 else f'⚠️  {r.status_code}'
    except Exception as e:
        status = f'❌ {e}'
    print(f'{status}  {name}')

✅  LiteLLM health
✅  LiteLLM models
✅  Ollama version
✅  Ollama models
✅  n8n health


## 2. List Available LiteLLM Models

In [3]:
r = requests.get(f'{LITELLM_URL}/v1/models', headers=HEADERS)
models = r.json().get('data', [])
print(f'Total models registered: {len(models)}\n')
for m in sorted(models, key=lambda x: x['id']):
    print(f'  {m["id"]}')

Total models registered: 87

  _claude-haiku
  _claude-opus
  _claude-sonnet
  _codestral
  _deepseek-chat
  _deepseek-reasoner
  _gemini-flash
  _gemini-flash-25
  _gemini-pro
  _gpt-4o
  _gpt-4o-mini
  _groq-deepseek-r1
  _groq-llama3-70b
  _groq-llama3-8b
  _hf-zephyr
  _mistral-large
  _mistral-small
  _o3
  _o4-mini
  _ollama-cloud-llama3
  _ollama-cloud-mistral
  _ollama-codellama
  _ollama-deepseek
  _ollama-fast
  _ollama-llama3
  _ollama-mistral
  _openrouter-qwen3
  _perplexity-sonar
  _sambanova-405b
  _venice-admin
  _venice-llama3
  _venice-mistral
  cloud/chat
  cloud/code
  cloud/fast
  cloud/reason
  cloud/search
  cloud/smart
  free/chat
  free/code
  free/dolphin-mistral-24b-venice-edition:free
  free/fast
  free/free
  free/gemini-2.0-flash
  free/gemini-2.0-flash-lite
  free/gemma-3-12b-it:free
  free/gemma-3-27b-it:free
  free/gemma-3-4b-it:free
  free/gemma-3n-e2b-it:free
  free/gemma-3n-e4b-it:free
  free/glm-4.5-air:free
  free/gpt-oss-120b:free
  free/gpt-oss-2

## 3. List Ollama Local Models

In [4]:
r = requests.get(f'{OLLAMA_URL}/api/tags')
models = r.json().get('models', [])
print(f'Local Ollama models: {len(models)}\n')
for m in models:
    size_gb = m.get('size', 0) / 1e9
    print(f'  {m["name"]:45s}  {size_gb:.1f} GB')

Local Ollama models: 6

  qwen2.5-coder:7b                               4.7 GB
  deepseek-r1:7b                                 4.7 GB
  mistral:7b-instruct-q4_0                       4.1 GB
  llama3.1:8b-instruct-q4_0                      4.7 GB
  llama3.2:3b                                    2.0 GB
  mistral:latest                                 4.4 GB


## 4. Test LiteLLM — Tier Router

In [5]:
def ask(prompt, model='hybrid/chat', max_tokens=500):
    """Send a prompt to LiteLLM and print the response."""
    payload = {
        'model': model,
        'max_tokens': max_tokens,
        'messages': [{'role': 'user', 'content': prompt}]
    }
    r = requests.post(f'{LITELLM_URL}/v1/chat/completions',
                      headers=HEADERS, json=payload, timeout=120)
    r.raise_for_status()
    data = r.json()
    content = data['choices'][0]['message']['content']
    model_used = data.get('model', 'unknown')
    usage = data.get('usage', {})
    print(f'Model  : {model_used}')
    print(f'Tokens : {usage.get("total_tokens", "?")} total')
    print(f'\n{content}')
    return data

# Quick smoke test across all tiers
# hybrid/fast tries Ollama first, falls back to cloud if Ollama is slow
_ = ask('Say hello in one sentence.', model='hybrid/fast')

Model  : hybrid/fast
Tokens : 51 total

Hello, it's nice to meet you.


In [6]:
# Test each tier group
# local/* tiers are tested via hybrid/* (tries Ollama, falls back to cloud)
tiers = ['hybrid/chat', 'hybrid/code', 'free/fast', 'cloud/fast']
prompt = 'What is 2 + 2? Reply in one word.'

for tier in tiers:
    try:
        r = requests.post(f'{LITELLM_URL}/v1/chat/completions',
            headers=HEADERS,
            json={'model': tier, 'max_tokens': 20,
                  'messages': [{'role': 'user', 'content': prompt}]},
            timeout=180)
        data = r.json()
        raw = data['choices'][0]['message'].get('content')
        answer = raw.strip() if raw else '[reasoning model — no content]'
        model  = data.get('model', tier)
        print(f'✅ {tier:20s} → {model:35s} → {answer}')
    except Exception as e:
        warn = 'ReadTimeout' in str(e) or 'timed out' in str(e).lower()
        icon = '⚠️ ' if warn else '❌'
        print(f'{icon} {tier:20s} → {e}')

✅ hybrid/chat          → hybrid/chat                         → Four.


✅ hybrid/code          → hybrid/code                         → 4


✅ free/fast            → free/fast                           → [reasoning model — no content]


✅ cloud/fast           → cloud/fast                          → Four.


In [7]:
# Run this in a new cell to see the raw error
r = requests.post(f'{LITELLM_URL}/v1/chat/completions',
    headers=HEADERS,
    json={'model': 'cloud/fast', 'max_tokens': 20,
          'messages': [{'role': 'user', 'content': 'What is 2+2?'}]},
    timeout=30)
print(r.status_code)
print(r.json())

200
{'id': 'chatcmpl-33365e7e-29f4-4a1e-9c9b-9485e150e253', 'created': 1773196355, 'model': 'cloud/fast', 'object': 'chat.completion', 'choices': [{'finish_reason': 'stop', 'index': 0, 'message': {'content': '2 + 2 = 4', 'role': 'assistant', 'provider_specific_fields': {'citations': None, 'thinking_blocks': None}}}], 'usage': {'completion_tokens': 13, 'prompt_tokens': 14, 'total_tokens': 27, 'completion_tokens_details': {'reasoning_tokens': 0, 'text_tokens': 13}, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_creation_tokens': 0, 'cache_creation_token_details': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available'}}


## 5. Inspect LiteLLM Spend Logs (PostgreSQL)

In [8]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(DB_URL)

with engine.connect() as conn:
    df = pd.read_sql(text("""
        SELECT
            DATE_TRUNC('hour', "startTime") as hour,
            model,
            COUNT(*)                        as requests,
            SUM(total_tokens)               as tokens,
            SUM(spend)                      as cost_usd
        FROM "LiteLLM_SpendLogs"
        WHERE "startTime" > NOW() - INTERVAL '24 hours'
        GROUP BY 1, 2
        ORDER BY 1 DESC, 4 DESC
    """), conn)

pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.6f}'.format)
print(f'Requests in last 24h: {df["requests"].sum()}')
print(f'Total tokens        : {df["tokens"].sum():,}')
print(f'Total cost (USD)    : ${df["cost_usd"].sum():.6f}\n')
df

Requests in last 24h: 205
Total tokens        : 91,418
Total cost (USD)    : $0.016481



,hour,model,requests,tokens,cost_usd
0,2026-03-11 02:00:00,ollama_chat/qwen2.5-coder:7b,3,3350,0.000000
1,2026-03-11 02:00:00,ollama_chat/llama3.1:8b-instruct-q4_0,4,2979,0.000000
2,2026-03-11 02:00:00,openrouter/qwen/qwen3-coder:free,6,2479,0.000000
3,2026-03-11 02:00:00,groq/llama-3.1-8b-instant,4,240,0.000014
4,2026-03-11 01:00:00,openrouter/qwen/qwen3-coder:free,7,4040,0.000000
...,...,...,...,...,...
56,2026-03-10 12:00:00,openrouter/stepfun/step-3.5-flash:free,2,140,0.000000
57,2026-03-10 12:00:00,groq/llama-3.1-8b-instant,2,92,0.000005
58,2026-03-10 12:00:00,openai/gpt-4o-mini,1,0,0.000000
59,2026-03-10 12:00:00,anthropic/claude-haiku-4-5-20251001,1,0,0.000000


## 6. Free Model Sync — Manual Trigger & Audit

In [9]:
# Run the free model sync script and capture output
import subprocess, sys

env = os.environ.copy()
env['LITELLM_BASE_URL'] = LITELLM_URL

result = subprocess.run(
    [sys.executable, '/home/jovyan/scripts/free_model_sync.py', '--dry-run', '--verbose'],
    capture_output=True, text=True, env=env
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)


STDERR: 2026-03-11 02:32:35 [INFO] ============================================================
2026-03-11 02:32:35 [INFO] Starting free model sync
2026-03-11 02:32:35 [INFO] ============================================================
2026-03-11 02:32:35 [INFO] Fetching OpenRouter model catalog...
2026-03-11 02:32:35 [INFO] OpenRouter: found 27 free models
2026-03-11 02:32:35 [INFO] Verifying Groq free models...
2026-03-11 02:32:35 [INFO] Groq: found 2 free models
2026-03-11 02:32:35 [INFO] Fetching Gemini model list...
2026-03-11 02:32:35 [INFO] Gemini: found 2 free models
2026-03-11 02:32:35 [INFO] Total free models discovered: 31
2026-03-11 02:32:36 [INFO] Currently registered free models: 35
2026-03-11 02:32:36 [INFO] To add:    0
2026-03-11 02:32:36 [INFO] To remove: 0
2026-03-11 02:32:36 [INFO] Syncing free tier group aliases...
2026-03-11 02:32:36 [INFO] [DRY RUN] Would ADD: free/chat → openrouter/qwen/qwen3-next-80b-a3b-instruct:free
2026-03-11 02:32:36 [INFO] [DRY RUN] Would

## 7. Scratch — Ad-hoc Testing

In [10]:
# Use this cell for one-off tests
# Example: test a specific model directly
try:
    _ = ask(
        prompt='Write a Python function that reverses a linked list.',
        model='free/code',
        max_tokens=800
    )
except Exception as e:
    print(f'⚠️  Scratch cell error (non-fatal): {e}')


⚠️  Scratch cell error (non-fatal): 429 Client Error: Too Many Requests for url: http://litellm:4000/v1/chat/completions
